# 5.4 各參數能用不同步幅嗎？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：旋鈕往哪邊轉，猜錯的代價會下降？**

藍線是 (w−3)²；橘點到綠點只走一小步。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/gradient.svg")))

**先想一想：**兩個梯度分別1和100，Adam第一步的更新也相差100倍嗎？

Adam同時追蹤方向平均和梯度大小。用平方平均作分母，讓不同旋鈕有不同有效步幅，但不能保證所有問題都更好。

**把直覺寫成數學：**m=β₁m+(1−β₁)g；v=β₂v+(1−β₂)g²；偏差修正後以 m̂/(√v̂+ε) 更新。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
g = torch.tensor([1.0, 100.0])
m = torch.zeros(2)
v = torch.zeros(2)
m = 0.9 * m + 0.1 * g
v = 0.999 * v + 0.001 * g.square()
m_hat, v_hat = m / (1 - 0.9), v / (1 - 0.999)
update = 0.001 * m_hat / (v_hat.sqrt() + 1e-8)
print("原梯度", g, "第一步更新", update)

**如何讀結果：**相似更新來自正規化；後面多步累積不必一直相同。

**只改一件事：**只把第二個梯度改成負數。
